In [1]:
# Spark Session
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Optimizing Joins")
    .master("spark://5892aedce22e:7077")
    .config("spark.cores.max", 16)
    .config("spark.executor.cores", 4)
    .config("spark.executor.memory", "512M")
    .getOrCreate()
)

spark

In [2]:
# Disable AQE and Broadcast join

spark.conf.set("spark.sql.adaptive.enabled", False)
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", False)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

# Join Big and Small table - SortMerge vs BroadCast Join

In [5]:
# Read EMP CSV data

_schema = "first_name string, last_name string, job_title string, dob string, email string, phone string, salary double, department_id int"

emp = spark.read.format("csv").schema(_schema).option("header", True).load("/data/input/employee_records.csv")

In [6]:
# Read DEPT CSV data

_dept_schema = "department_id int, department_name string, description string, city string, state string, country string"

dept = spark.read.format("csv").schema(_dept_schema).option("header", True).load("/data/input/department_data.csv")

In [ ]:
# Broadcast Join - Important Points
# ---------------------------------
# 1. Used when one table is small enough to fit in memory of each executor
# 2. Small table is broadcasted (copied) to all worker nodes
# 3. Avoids expensive shuffle of large table across cluster
# 4. Very fast compared to shuffle-based joins
# 5. Best suited for star schema joins (fact + dimension tables)
# 6. Controlled by spark.sql.autoBroadcastJoinThreshold (default ~10MB)
# 7. If table is too large, broadcast join may cause OutOfMemory error
# 8. Works well when one side is significantly smaller than the other
# 9. Reduces network I/O and improves performance

In [7]:
# Join Datasets

from pyspark.sql.functions import broadcast

#df_joined = emp.join(broadcast(dept), on=emp.department_id==dept.department_id, how="left_outer")
df_joined = emp.alias("e").join(broadcast(dept.alias("d")), on = "department_id", how = "left_outer")

In [8]:
df_joined.write.format("noop").mode("overwrite").save()

In [9]:
df_joined.explain()

== Physical Plan ==
*(2) Project [department_id#7, first_name#0, last_name#1, job_title#2, dob#3, email#4, phone#5, salary#6, department_name#17, description#18, city#19, state#20, country#21]
+- *(2) BroadcastHashJoin [department_id#7], [department_id#16], LeftOuter, BuildRight, false
   :- FileScan csv [first_name#0,last_name#1,job_title#2,dob#3,email#4,phone#5,salary#6,department_id#7] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/data/input/employee_records.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<first_name:string,last_name:string,job_title:string,dob:string,email:string,phone:string,s...
   +- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[0, int, false] as bigint)),false), [id=#74]
      +- *(1) Filter isnotnull(department_id#16)
         +- FileScan csv [department_id#16,department_name#17,description#18,city#19,state#20,country#21] Batched: false, DataFilters: [isnotnull(department_id#16)], Form

In [10]:
# Sort-Merge Join (SMJ) - Important Points
# ----------------------------------------
# 1. Default join strategy for large datasets in Spark
# 2. Used when both tables are large and cannot be broadcasted
# 3. Requires shuffle of both datasets across the cluster
# 4. Data is partitioned based on join keys after shuffle
# 5. Each partition is sorted by join key before joining
# 6. Join happens by merging two sorted datasets (like merge sort)
# 7. Efficient for large-scale distributed joins
# 8. More stable performance for big-big table joins
# 9. Expensive due to shuffle + sorting overhead
# 10. Uses more CPU and network compared to broadcast join
# 11. Preferred when cost-based optimization cannot use broadcast/hash join

In [11]:
df_joined = emp.join(dept, on=emp.department_id == dept.department_id, how="left_outer")

In [12]:
df_joined.write.format("noop").mode("overwrite").save()

In [13]:
df_joined.explain()

== Physical Plan ==
*(4) SortMergeJoin [department_id#7], [department_id#16], LeftOuter
:- *(1) Sort [department_id#7 ASC NULLS FIRST], false, 0
:  +- Exchange hashpartitioning(department_id#7, 200), ENSURE_REQUIREMENTS, [id=#155]
:     +- FileScan csv [first_name#0,last_name#1,job_title#2,dob#3,email#4,phone#5,salary#6,department_id#7] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/data/input/employee_records.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<first_name:string,last_name:string,job_title:string,dob:string,email:string,phone:string,s...
+- *(3) Sort [department_id#16 ASC NULLS FIRST], false, 0
   +- Exchange hashpartitioning(department_id#16, 200), ENSURE_REQUIREMENTS, [id=#167]
      +- *(2) Filter isnotnull(department_id#16)
         +- FileScan csv [department_id#16,department_name#17,description#18,city#19,state#20,country#21] Batched: false, DataFilters: [isnotnull(department_id#16)], Format: CSV, Location: I

# Join Big and Big table - SortMerge without Buckets

In [3]:
# Read Sales data

sales_schema = "transacted_at string, trx_id string, retailer_id string, description string, amount double, city_id string"

sales = spark.read.format("csv").schema(sales_schema).option("header", True).load("/data/input/new_sales.csv")

In [4]:
# Read City data

city_schema = "city_id string, city string, state string, state_abv string, country string"

city = spark.read.format("csv").schema(city_schema).option("header", True).load("/data/input/cities.csv")

In [5]:
# Join Data

df_sales_joined = sales.join(city, on=sales.city_id==city.city_id, how="left_outer")

In [6]:
df_sales_joined.write.format("noop").mode("overwrite").save()

In [7]:
# Explain Plan

df_sales_joined.explain()

== Physical Plan ==
*(4) SortMergeJoin [city_id#5], [city_id#12], LeftOuter
:- *(1) Sort [city_id#5 ASC NULLS FIRST], false, 0
:  +- Exchange hashpartitioning(city_id#5, 200), ENSURE_REQUIREMENTS, [id=#70]
:     +- FileScan csv [transacted_at#0,trx_id#1,retailer_id#2,description#3,amount#4,city_id#5] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/data/input/new_sales.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<transacted_at:string,trx_id:string,retailer_id:string,description:string,amount:double,cit...
+- *(3) Sort [city_id#12 ASC NULLS FIRST], false, 0
   +- Exchange hashpartitioning(city_id#12, 200), ENSURE_REQUIREMENTS, [id=#82]
      +- *(2) Filter isnotnull(city_id#12)
         +- FileScan csv [city_id#12,city#13,state#14,state_abv#15,country#16] Batched: false, DataFilters: [isnotnull(city_id#12)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/data/input/cities.csv], PartitionFilters: [], PushedFilters: [Is

# Write Sales and City data in Buckets

In [8]:
# Write Sales data in Buckets

sales.write.format("csv").mode("overwrite").bucketBy(4, "city_id").option("header", True).option("path", "/data/input/sales_bucket").saveAsTable("sales_bucket")

In [9]:
# Write City data in Buckets

city.write.format("csv").mode("overwrite").bucketBy(4, "city_id").option("header", True).option("path", "/data/input/datasets/city_bucket.csv").saveAsTable("city_bucket")

In [ ]:
# Sorting within bucket

# If you use .sortBy():

# df.write.bucketBy(4, "city_id").sortBy("city_id")

# Then each bucket is:
# sorted before writing
# improves range scans and joins further

In [10]:
# Check tables

spark.sql("show tables in default").show()

+---------+------------+-----------+
|namespace|   tableName|isTemporary|
+---------+------------+-----------+
|  default| city_bucket|      false|
|  default|sales_bucket|      false|
+---------+------------+-----------+



# Join Sales and City data - SortMerge with Bucket

In [11]:
# Read Sales table

sales_bucket = spark.read.table("sales_bucket")

In [12]:
# Read City table

city_bucket = spark.read.table("city_bucket")

In [13]:
# Join datasets

df_joined_bucket = sales_bucket.join(city_bucket, on=sales_bucket.city_id==city_bucket.city_id, how="left_outer")

In [16]:
# Bucketing is a technique in Spark that is used to distribute data across multiple buckets or files based on the hash of a column value.
# This method is particularly useful when working with large datasets and performing operations like joins, which can be computationally expensive.

# Bucketing works by specifying a column and a number of buckets during the creation of the DataFrame.
# Spark then applies a hash function to the specified column and divides the data into buckets corresponding to the hash values.
# Hashing is applied for each row:
# bucket_id = hash(bucket_column) % num_buckets
# Example: user_id = 42 → hash(42) → 3 → bucket 3
# Rows are grouped by bucket id; Spark internally creates logical groups of rows per bucket.
# Sorting within bucket if you use .sortBy()
# Writing to disk (physical storage); Spark writes one file per bucket per partition. (Each Spark partition independently writes data, and for every bucket it produces, it may create a separate file—so buckets are not single files globally, but grouped outputs across partitions.)
# The number of buckets remains fixed, so the distribution of data doesn’t change with the size of the data.

In [19]:
# Write dataset

df_joined_bucket.write.format("noop").mode("overwrite").save()

# by default, Task 0 (the “first task”) in Apache Spark will read all data in partition/bucket 0

In [20]:
df_joined_bucket.explain()

== Physical Plan ==
*(3) SortMergeJoin [city_id#124], [city_id#131], LeftOuter
:- *(1) Sort [city_id#124 ASC NULLS FIRST], false, 0
:  +- FileScan csv default.sales_bucket[transacted_at#119,trx_id#120,retailer_id#121,description#122,amount#123,city_id#124] Batched: false, Bucketed: true, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/data/input/sales_bucket], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<transacted_at:string,trx_id:string,retailer_id:string,description:string,amount:double,cit..., SelectedBucketsCount: 4 out of 4
+- *(2) Sort [city_id#131 ASC NULLS FIRST], false, 0
   +- *(2) Filter isnotnull(city_id#131)
      +- FileScan csv default.city_bucket[city_id#131,city#132,state#133,state_abv#134,country#135] Batched: false, Bucketed: true, DataFilters: [isnotnull(city_id#131)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/data/input/datasets/city_bucket.csv], PartitionFilters: [], PushedFilters: [IsNotNull(city_id)], ReadSc

In [ ]:
# Spark Bucketing and Shuffle Notes

# 1. Join column different from bucket column
#    Full shuffle on both tables

# 2. Join column same but only one table bucketed
#    Shuffle required (at least non-bucketed side, often both)

# 3. Join column same but different number of buckets
#    Shuffle required (repartition one or both sides)

# 4. Join column same and same number of buckets
#    No shuffle (best case, optimized join)

# Edge cases:

# 5. Bucketing metadata not recognized
#    Full shuffle (bucketing ignored)

# 6. Data not sorted within buckets
#    May require additional sort (sometimes shuffle)

# 7. Adaptive Query Execution (AQE)
#    May introduce shuffle by changing execution plan

# 8. Bucket pruning with filter on bucket column
#    Reduces data read, does not affect shuffle directly

# Mental model:
# Bucketing is pre-partitioned data on disk, used only if all conditions align